In [ ]:
%pip install kagglehub --quiet

import struct
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

dataset_path = Path(kagglehub.dataset_download("hojjatk/mnist-dataset"))


def find_file(*names):
    for name in names:
        matches = list(dataset_path.rglob(name))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Не найден файл: {names}")


def read_images(file_path):
    with open(file_path, "rb") as file:
        magic, count, rows, columns = struct.unpack(">IIII", file.read(16))
        if magic != 2051:
            raise ValueError(f"Некорректный IDX-файл изображений: {file_path}")
        images = np.frombuffer(file.read(), dtype=np.uint8)
    return images.reshape(count, rows * columns)


def read_labels(file_path):
    with open(file_path, "rb") as file:
        magic, count = struct.unpack(">II", file.read(8))
        if magic != 2049:
            raise ValueError(f"Некорректный IDX-файл меток: {file_path}")
        labels = np.frombuffer(file.read(), dtype=np.uint8)
    return labels.reshape(count)


x_train_full = read_images(find_file("train-images.idx3-ubyte", "train-images-idx3-ubyte"))
y_train_full = read_labels(find_file("train-labels.idx1-ubyte", "train-labels-idx1-ubyte"))
x_test_full = read_images(find_file("t10k-images.idx3-ubyte", "t10k-images-idx3-ubyte"))
y_test_full = read_labels(find_file("t10k-labels.idx1-ubyte", "t10k-labels-idx1-ubyte"))

print(f"Обучающая выборка: {x_train_full.shape}")
print(f"Тестовая выборка: {x_test_full.shape}")
print("Количество изображений каждого класса:")
for digit, count in enumerate(np.bincount(y_train_full, minlength=10)):
    print(f"{digit}: {count}")

train_indices, _ = train_test_split(
    np.arange(len(y_train_full)),
    train_size=10000,
    random_state=42,
    stratify=y_train_full,
)
test_indices, _ = train_test_split(
    np.arange(len(y_test_full)),
    train_size=3000,
    random_state=42,
    stratify=y_test_full,
)

x_train = x_train_full[train_indices].astype(np.float32) / 255.0
y_train = y_train_full[train_indices]
x_test = x_test_full[test_indices].astype(np.float32) / 255.0
y_test = y_test_full[test_indices]

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        n_jobs=-1,
        random_state=42,
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=60,
        learning_rate=0.1,
        max_leaf_nodes=15,
        random_state=42,
    ),
}

results = {}
for name, model in models.items():
    model.fit(x_train, y_train)
    predictions = model.predict(x_test)
    results[name] = accuracy_score(y_test, predictions)
    print(f"{name}: accuracy={results[name]:.4f}")

best_model = max(results, key=results.get)
print(f"Лучшая модель: {best_model}, accuracy={results[best_model]:.4f}")

plt.figure(figsize=(8, 4))
plt.bar(results.keys(), results.values(), color=["steelblue", "seagreen", "darkorange"])
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
